
# PIANO AMT(ONSET AND FRAME) — Public Colab Demo

This notebook is the **public demo** for the project. It is designed to feel like a polished Magenta-style demo notebook, while using your own PyTorch Onsets & Frames implementation and the repo-side `demo/` helper layer.

## What this notebook does

1. **Setup**
   - clones the public repo
   - installs demo dependencies
   - downloads the public checkpoint from Hugging Face
   - loads the model and prints checkpoint transparency information
2. **Input source**
   - upload custom audio **or**
   - choose a small curated public MAESTRO demo sample
3. **Transcription**
   - runs the model once
   - stores raw outputs: onset / frame / offset / velocity
4. **Live decoding controls**
   - onset threshold slider
   - frame threshold slider
   - slider changes re-decode from stored predictions without rerunning the model
5. **Results**
   - original audio
   - synthesized MIDI audio
   - interactive MIDI plot (Bokeh / Visual MIDI)
   - predicted vs GT piano-roll plots
   - diff plot (green match, red extra, blue missed)
6. **Evaluation**
   - Frame F1
   - Note F1 (onset + pitch)
   - Note w/ offset F1
   - Note w/ offset + velocity F1
   - failure analysis
7. **Downloads**
   - `.mid`
   - `.png`
   - `.html`

> Before publishing this notebook, update the repo URL and branch in the configuration cell below.


In [ ]:

#@title 0. Public demo configuration (edit before publishing)
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  #@param {type:"string"}
REPO_BRANCH = "main"  #@param {type:"string"}
REPO_CLONE_DIR = "/content/repo"  #@param {type:"string"}

# Optional overrides for the demo layer.
HF_REPO_ID_OVERRIDE = "YOUR_USERNAME/piano-amt-demo"  #@param {type:"string"}
HF_CHECKPOINT_FILENAME_OVERRIDE = "checkpoints/best.pt"  #@param {type:"string"}
MODEL_COMPLEXITY_OVERRIDE = 48  #@param {type:"integer"}


In [ ]:

#@title 1. Clone repo and install public demo requirements
import os
from pathlib import Path

repo_dir = Path(REPO_CLONE_DIR)
if not repo_dir.exists():
    !git clone -q --branch "$REPO_BRANCH" "$REPO_URL" "$REPO_CLONE_DIR"
else:
    print(f"Repo already present at {repo_dir}")

%cd $REPO_CLONE_DIR
!pip -q install -r requirements_demo.txt


In [ ]:

#@title 2. Imports and runtime setup
import os
import sys
import uuid
from pathlib import Path

import pandas as pd
import torch
import ipywidgets as widgets
from IPython.display import Audio, FileLink, Markdown, clear_output, display

# Let the helper layer read the public HF repo/checkpoint details from env vars.
os.environ["AMT_DEMO_HF_REPO_ID"] = HF_REPO_ID_OVERRIDE
os.environ["AMT_DEMO_HF_CHECKPOINT_FILENAME"] = HF_CHECKPOINT_FILENAME_OVERRIDE
os.environ["AMT_DEMO_MODEL_COMPLEXITY"] = str(MODEL_COMPLEXITY_OVERRIDE)
os.environ["AMT_DEMO_REPO_ROOT"] = REPO_CLONE_DIR

if REPO_CLONE_DIR not in sys.path:
    sys.path.insert(0, REPO_CLONE_DIR)

from demo.checkpoint import load_demo_model, format_checkpoint_summary, maybe_torchinfo_summary
from demo.demo_config import (
    DEFAULT_FRAME_THRESHOLD,
    DEFAULT_ONSET_THRESHOLD,
    HTML_DIR,
    MIDI_DIR,
    PLOT_DIR,
    ensure_demo_dirs,
)
from demo.evaluation import evaluate_prediction_vs_gt, failure_table, main_scores_table
from demo.inference import decode_prediction_to_pretty_midi, run_model_on_mel, save_prediction_midi
from demo.rendering import (
    plot_piano_roll_side_by_side,
    plot_roll_diff,
    render_bokeh_midi,
    synthesize_pretty_midi,
)
from demo.sources import (
    audio_path_to_mel,
    list_demo_sample_names,
    load_ground_truth_labels,
    resolve_demo_sample_paths,
    save_uploaded_audio,
)

ensure_demo_dirs()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:

#@title 3. Load checkpoint and show transparency info
MODEL, CKPT, CKPT_PATH = load_demo_model(DEVICE)

print(format_checkpoint_summary(MODEL, CKPT, CKPT_PATH))
print("\nCompact torchinfo summary:\n")
print(maybe_torchinfo_summary(MODEL))
print("\nFull architecture:\n")
print(MODEL)


In [ ]:

#@title 4. Build the interactive demo controls
DEMO_SAMPLE_NAMES = list_demo_sample_names()

state = {
    "source_type": None,
    "title": None,
    "audio_path": None,
    "mel": None,
    "pred": None,
    "gt": None,
    "midi_path": None,
    "png_path": None,
    "html_path": None,
}

source_selector = widgets.ToggleButtons(
    options=["Upload custom audio", "Select public MAESTRO demo sample"],
    description="Source:",
    style={"description_width": "initial"},
)

upload_widget = widgets.FileUpload(
    accept=".wav,.mp3,.flac,.ogg,.m4a,.aiff,.aif,.aac",
    multiple=False,
    description="Upload audio",
)

sample_dropdown = widgets.Dropdown(
    options=DEMO_SAMPLE_NAMES if DEMO_SAMPLE_NAMES else ["<no demo samples found>"],
    description="Demo sample:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="700px"),
)

onset_slider = widgets.FloatSlider(
    value=DEFAULT_ONSET_THRESHOLD,
    min=0.30,
    max=0.90,
    step=0.01,
    description="Onset thr:",
    readout_format=".2f",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px"),
)

frame_slider = widgets.FloatSlider(
    value=DEFAULT_FRAME_THRESHOLD,
    min=0.30,
    max=0.90,
    step=0.01,
    description="Frame thr:",
    readout_format=".2f",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px"),
)

run_button = widgets.Button(description="Run transcription", button_style="success", icon="play")
status_out = widgets.Output()
results_out = widgets.Output()

upload_box = widgets.VBox([widgets.HTML("<b>Upload custom piano audio</b>"), upload_widget])
sample_box = widgets.VBox([widgets.HTML("<b>Select a public MAESTRO demo sample</b>"), sample_dropdown])


def refresh_source_visibility(*_):
    if source_selector.value == "Upload custom audio":
        upload_box.layout.display = "block"
        sample_box.layout.display = "none"
    else:
        upload_box.layout.display = "none"
        sample_box.layout.display = "block"

refresh_source_visibility()
source_selector.observe(refresh_source_visibility, names="value")

controls = widgets.VBox([
    source_selector,
    upload_box,
    sample_box,
    widgets.HTML("<b>Live decode thresholds</b>"),
    onset_slider,
    frame_slider,
    run_button,
])

display(controls, status_out, results_out)


In [ ]:

#@title 5. Demo logic (run once, then re-decode live from stored predictions)
def _extract_upload():
    if not upload_widget.value:
        raise ValueError("Please upload an audio file first.")

    item = upload_widget.value[0] if isinstance(upload_widget.value, tuple) else next(iter(upload_widget.value.values()))
    if isinstance(item, dict):
        name = item.get("name", "uploaded_audio.wav")
        content = bytes(item["content"])
    else:
        raise ValueError("Unsupported FileUpload format in this environment.")
    return name, content


def _prepare_input_from_upload():
    name, content = _extract_upload()
    audio_path = save_uploaded_audio(content, name)
    mel = audio_path_to_mel(audio_path)
    return {
        "source_type": "custom_upload",
        "title": Path(audio_path).stem,
        "audio_path": audio_path,
        "mel": mel,
        "gt": None,
    }


def _prepare_input_from_sample():
    audio_path, labels_path = resolve_demo_sample_paths(sample_dropdown.value, REPO_CLONE_DIR)
    mel = audio_path_to_mel(audio_path)
    gt = load_ground_truth_labels(labels_path)
    return {
        "source_type": "demo_sample",
        "title": Path(audio_path).stem,
        "audio_path": audio_path,
        "mel": mel,
        "gt": gt,
    }


def _render_current_result():
    if state["pred"] is None:
        return

    onset_threshold = float(onset_slider.value)
    frame_threshold = float(frame_slider.value)
    pred = state["pred"]
    gt = state["gt"]
    title = state["title"]
    audio_path = state["audio_path"]

    with results_out:
        clear_output(wait=True)

        run_id = f"{title}_{uuid.uuid4().hex[:8]}"
        midi_path = MIDI_DIR / f"{run_id}.mid"
        png_path = PLOT_DIR / f"{run_id}_roll.png"
        diff_path = PLOT_DIR / f"{run_id}_diff.png"
        html_path = HTML_DIR / f"{run_id}_midi_plot.html"

        pm = decode_prediction_to_pretty_midi(pred, onset_threshold, frame_threshold)
        midi_audio = synthesize_pretty_midi(pm)
        save_prediction_midi(pred, midi_path, onset_threshold, frame_threshold)

        state["midi_path"] = midi_path
        state["png_path"] = png_path
        state["html_path"] = html_path

        display(Markdown(f"## Results — {title}"))
        display(Markdown(f"**Source:** `{state['source_type']}`"))
        display(Markdown(f"**Thresholds:** onset=`{onset_threshold:.2f}` | frame=`{frame_threshold:.2f}`"))

        display(Markdown("### Audio comparison"))
        print("Original audio")
        display(Audio(filename=str(audio_path)))
        print("Synthesized predicted MIDI")
        display(Audio(midi_audio, rate=16000))

        display(Markdown("### Interactive MIDI view"))
        plot = render_bokeh_midi(pm, html_path=html_path, show_inline=True)
        if plot is None:
            print("visual_midi / bokeh unavailable in this environment; HTML export skipped.")

        display(Markdown("### Piano-roll visualisation"))
        fig = plot_piano_roll_side_by_side(
            pred_frame=pred["frame"],
            gt_frame=None if gt is None else gt["frame"],
            title=title,
            save_path=png_path,
        )
        display(fig)

        if gt is not None:
            display(Markdown("### Diff plot"))
            diff_fig = plot_roll_diff(
                pred_frame=pred["frame"],
                gt_frame=gt["frame"],
                frame_threshold=frame_threshold,
                save_path=diff_path,
            )
            display(diff_fig)

            results = evaluate_prediction_vs_gt(pred, gt, onset_threshold, frame_threshold)
            display(Markdown("### Quantitative evaluation"))
            display(main_scores_table(results).style.format({"Value": "{:.4f}"}))

            display(Markdown("### Failure-case analysis"))
            display(failure_table(results).style.format({"Value": "{:.4f}"}))
        else:
            display(Markdown("> No ground-truth labels are available for uploaded custom audio, so live metrics and failure analysis are only shown for public demo samples."))

        display(Markdown("### Downloads"))
        print("MIDI file:")
        display(FileLink(str(midi_path)))
        print("Piano-roll PNG:")
        display(FileLink(str(png_path)))
        if html_path.exists():
            print("Interactive MIDI HTML:")
            display(FileLink(str(html_path)))


def _run_demo(_=None):
    with status_out:
        clear_output(wait=True)
        print("Preparing input and running model inference...")

    try:
        prepared = _prepare_input_from_upload() if source_selector.value == "Upload custom audio" else _prepare_input_from_sample()
        pred = run_model_on_mel(MODEL, prepared["mel"], DEVICE)

        state.update(prepared)
        state["pred"] = pred

        with status_out:
            clear_output(wait=True)
            print("Inference complete.")
            print("Title:", prepared["title"])
            print("Mel shape:", tuple(prepared["mel"].shape))
            print("Prediction tensor shapes:", {k: tuple(v.shape) for k, v in pred.items()})

        _render_current_result()
    except Exception as exc:
        with status_out:
            clear_output(wait=True)
            print("ERROR:", exc)
            raise


def _rerender_from_sliders(change):
    if change["name"] == "value" and state["pred"] is not None:
        _render_current_result()

run_button.on_click(_run_demo)
onset_slider.observe(_rerender_from_sliders, names="value")
frame_slider.observe(_rerender_from_sliders, names="value")



## How to use this notebook

1. Run the setup cells from top to bottom.
2. Choose **Upload custom audio** or **Select public MAESTRO demo sample**.
3. Click **Run transcription**.
4. Move the **onset** and **frame** threshold sliders.
5. Notice that the notebook **re-decodes** the stored predictions without rerunning the neural network.

This behaviour is important because it makes the post-processing stage explicit and inspectable.
